In [ ]:
# Imports
import os
import ast
import json
from uuid import uuid4
import re
import requests
from typing import List, Dict
import neo4j
import dotenv

dotenv.load_dotenv()

# Define the root directory where repositories exist
SRC_PATH = "../../src/"

# GPT-4 API for concise description generation
GPT4_URL = "https://routes.prod.dart.prv.vai-compute.com/routes/v1/gpt-4/prompt"
# Embedding API
EMBEDDING_URL = "https://ollama-2.prod.dart.prv.vai-compute.com/api/embed"

output_dir = "../../src/jupyter/tools/"
tools_script_path = os.path.join(output_dir, "tools.py")

exclude_list = [
    "transformer_embed",
    "transformer_prompt",
    "crawler_extract",
    "crawler_warc_to_html",
    "routes_models",
    "scheduler_minio_download_object",
    "scheduler_minio_download_prefix",
]

In [2]:
# Reusable functions

def make_api_request(url: str, payload: dict) -> dict:
    """
    Makes a POST request to the specified URL with the given payload.
    """
    try:
        # Send the POST request
        response = requests.post(url, json=payload)
        response.raise_for_status()  # Raise an exception for HTTP errors
        return response.json()
    except requests.RequestException as e:
        # Handle request errors and print the error message
        print(f"Error: Failed to make API request to {url}. Error: {e}")
        return {}

## 1. Metadata Creation

In [3]:
# Scan Repos, Generate Tool Descriptions, and Create Tool Embeddings

def scan_repos_for_api(debug=False) -> List[dict]:
    """
    Scans all repositories inside the `src/` directory and extracts API metadata for tool registration.

    Args:
        debug (bool): If True, enables debug mode for faster processing by skipping certain steps.

    Returns:
        List[dict]: A list of metadata dictionaries for all API functions found in the repositories.
    """
    all_api_metadata = []
    for repo_name in os.listdir(SRC_PATH):
        repo_path = os.path.join(SRC_PATH, repo_name)
        if os.path.isdir(repo_path):  # Ensure it's a valid repository folder
            for root, _, files in os.walk(repo_path):
                for file in files:
                    if file == "api.py":  # Target only `api.py` files
                        file_path = os.path.join(root, file)
                        api_metadata = extract_api_metadata(file_path, repo_name, debug=debug)
                        all_api_metadata.extend(api_metadata)
    return all_api_metadata


def generate_tool_description(tool_name: str) -> str:
    """
    Generates a concise description for a tool using GPT-4.
    """
    payload = {
        "request_type": "chat",
        "prompt": f"Describe the functionality of '{tool_name}' in a **single concise sentence**. "
                  f"Start the description directly with the action, avoiding phrases like "
                  f"'{tool_name} is a function that...'"
    }
    response = make_api_request(GPT4_URL, payload)
    return response.get("summary", "No description available")


def generate_tool_embedding(description: str) -> List[float]:
    """
    Generates an embedding for a tool description.
    """
    payload = {"model": "bge-m3", "input": description}
    response = make_api_request(EMBEDDING_URL, payload)
    return response.get("embeddings", [[]])[0]  # Extract the first embedding


def _extract_fixed_params(node: ast.FunctionDef) -> Dict[str, str]:
    """
    Extracts fixed parameters from `data.get(...)` or `data.pop(...)` calls in a function body,
    but only includes parameters without a default value.
    """
    fixed_params = {}
    for sub_node in ast.walk(node):
        if isinstance(sub_node, ast.Call) and isinstance(sub_node.func, ast.Attribute):
            # Check if the function is `get` or `pop`
            if sub_node.func.attr in {"get", "pop"} and isinstance(sub_node.func.value, ast.Name):
                # Extract the key name
                key_name = sub_node.args[0].value if len(sub_node.args) > 0 and isinstance(sub_node.args[0], ast.Constant) else None
                # Check if a default value is provided
                has_default = len(sub_node.args) > 1 or sub_node.keywords
                # Only include if no default value is provided
                if key_name and not has_default:
                    fixed_params[key_name] = None
    return fixed_params

In [4]:
# Extract API Metadata and Generate Tool Definitions
def extract_api_metadata(api_file: str, repo_name: str, debug=False) -> List[dict]:
    """Extract metadata for API functions from api.py files in repositories."""
    with open(api_file, "r", encoding="utf-8") as f:
        source_code = f.read()

    tree = ast.parse(source_code)
    api_metadata = []

    for i, node in enumerate(tree.body):
        if isinstance(node, ast.FunctionDef):  # Look for function definitions
            function_name = node.name
            tool_name = f"{repo_name}_{function_name}"  # Prefix tool names
            output_file = os.path.join(output_dir, f"{tool_name}.json")

            # Skip functions if "healthcheck" is in the function name
            if "healthcheck" in function_name.lower() or function_name.startswith("_"):
                continue
            if debug and api_metadata != []:
                continue  # Speed it up
            if function_name[0] == "_":
                continue
            if os.path.exists(output_file):
                continue
            if tool_name in exclude_list:
                continue

            # Skip functions if "healthcheck" is in the route or function name
            route_match = re.search(r'@[\w_]+\.route\(["\'](.+?)["\']', source_code)
            if route_match:
                full_url = f"https://{repo_name}.prod.dart.prv.vai-compute.com/{repo_name}/v1/{function_name}"

                # Extract HTTP method
                http_method_match = re.search(r'methods=\[["\'](\w+)["\']\]', source_code)
                http_method = http_method_match.group(1) if http_method_match else "GET"

                # Generate tool description & embedding
                tool_description = generate_tool_description(tool_name)
                tool_description_embedding = generate_tool_embedding(tool_description)

                # Extract function arguments and defaults
                body_params = _extract_fixed_params(node)

                # Handle default values for arguments
                if node.args.defaults:
                    num_defaults = len(node.args.defaults)
                    for arg, default in zip(node.args.args[-num_defaults:], node.args.defaults):
                        body_params[arg.arg] = ast.literal_eval(default)

                # Define fixed parameters
                fixed_params = {
                    "url": full_url,
                    "headers": {},
                    "body": {},
                }

                required_params = [key for key, value in body_params.items() if value is None]

                # Dynamically construct prop_dict
                docstring = ast.get_docstring(node)
                data = parse_docstring(docstring, tool_name)
                prop_dict = create_prop_dict(data, docstring)

                # Create tool definition dynamically
                tool_definition = {
                    "type": "function",
                    "function": {
                        "name": tool_name,  # Use descriptive tool_name
                        "description": tool_description,
                        "strict": True,
                        "parameters": {
                            "type": "object",
                            "properties": prop_dict,
                            "required": required_params,
                            "additionalProperties": False,
                        },
                    },
                }

                if "_" in tool_name:
                    name_parts = tool_name.split("_")[1:]  # Remove the repo name prefix
                    if name_parts and name_parts[0] in {"get", "post"}:
                        name_parts.pop(0)  # Remove "get" or "post" if present
                    name = " ".join(name_parts) if name_parts else tool_name
                else:
                    name = tool_name
                name = name.capitalize()  # Ensure name is capitalized

                # Store tool metadata
                tool_metadata = {
                    "uuid": str(uuid4()),
                    "name": name,
                    "tool_name": tool_name,
                    "step_type": "http_post",
                    "tool_fixed_params": json.dumps(fixed_params),
                    "tool_definition": json.dumps(tool_definition),
                    "tool_description": tool_description if not debug else None,
                    "tool_description_embedding": tool_description_embedding if not debug else None,
                }

                # Save the tool metadata to a file
                output_file = os.path.join(output_dir, f"{tool_name}.json")
                with open(output_file, "w") as f:
                    json.dump(tool_metadata, f, indent=4)


                # Append to the list of all metadata
                api_metadata.append(tool_metadata)
                # print(f"Tool metadata saved to {output_file}")

    return api_metadata


In [5]:
def parse_docstring(docstring: str, tool_name) -> dict:
    """Parse the JSON-like example in a docstring and convert it into a Python dictionary."""
    # Simplified regex to find the first { and the last }
    print(f"DEBUG: {tool_name}")
    match = re.search(r"\{(.*)\}", docstring, re.DOTALL)
    if not match:
        return {}  # Return an empty dictionary if no match is found

    json_string = match.group(0)  # Extract the full JSON-like block

    # Step 1: Remove ellipses and normalize whitespace
    json_string = re.sub(r"\.\.\.", "", json_string)  # Remove ellipses
    json_string = re.sub(r"\s+", " ", json_string)  # Normalize whitespace

    # Step 2: Replace Python-style booleans with JSON-compliant booleans
    json_string = json_string.replace("True", "true").replace("False", "false")

    # Step 3: Parse the JSON string
    try:
        json_data = json.loads(json_string)  # Convert JSON string to Python dictionary
    except json.JSONDecodeError as e:
        # Debugging failed parsing attempt
        raise ValueError(f"Invalid JSON format in docstring: {e}")

    return json_data

# Generate metadata for arguments
def create_args(data: dict, docstring: str) -> dict:
    """
    Extract top-level scalar arguments (non-nested values) from the payload
    and generate metadata for each argument.

    Args:
        data (dict): The payload to process.

    Returns:
        dict: A dictionary where each key is an argument name, and the value
              is a dictionary containing metadata such as "type" and "description".
    """
    def infer_type(value):
        if isinstance(value, str):
            return "string"
        elif isinstance(value, bool):
            return "boolean"
        elif isinstance(value, int):
            return "integer"
        elif isinstance(value, float):
            return "number"
        else:
            return "string"  # Default to string if type is ambiguous

    return {
        key: {
            "type": infer_type(value),
            "description": get_arg_description(key, docstring)
        }
        for key, value in data.items()
        if not isinstance(value, (list, dict))  # Exclude arrays and nested objects
    }

# Generate metadata for arrays
def create_array_properties(data: dict, docstring: str) -> dict:
    """
    Extract array properties from the payload.

    Args:
        data (dict): The payload to process.

    Returns:
        dict: A dictionary of array properties with metadata.
    """
    return {
        key: {
            "type": "array",
            "items": {"type": "number" if all(isinstance(i, (int, float)) for i in value) else "string"},
            "description": get_arg_description(key, docstring)
        }
        for key, value in data.items()
        if isinstance(value, list)
    }

# Generate metadata for nested objects
def create_object_properties(data: dict, docstring: str) -> dict:
    """
    Extract nested object properties from the payload.

    Args:
        data (dict): The payload to process.

    Returns:
        dict: A dictionary of object properties with metadata.
    """
    return {
        key: {
            "type": "object",
            "properties": {
                **create_args(value, docstring),  # Handle scalar arguments
                **create_array_properties(value, docstring),  # Handle arrays
                **create_object_properties(value, docstring),  # Handle nested objects recursively
            },
            "required": list(value.keys()),  # Mark all keys as required for now
        }
        for key, value in data.items()
        if isinstance(value, dict)  # Only process nested dictionaries
    }

def create_prop_dict(data: dict, docstring: str) -> dict:
    """
    Create a property dictionary for the tool definition.

    Args:
        data (dict): The payload to process.
        fixed_params (dict): Fixed parameters to include in the properties.

    Returns:
        dict: A dictionary of properties with metadata.
    """
    # Create a dictionary of properties
    return {
        **create_args(data, docstring),
        **create_array_properties(data, docstring),
        **create_object_properties(data, docstring),
    }

def get_arg_description(key: str, docstring: str) -> dict:
    prompt = (
        "Concisely describe this key\n"
        f"{key}\n\n"
        "Use the following docstring as context:\n"
        f"{docstring}\n\n"
        "Only give the description. Do not include comments or explanations.\n"
        "Your response should be a half of a sentence like <var> indicates ... or <var> does ....\n"
    )
    payload = {
        "request_type": "chat",
        "prompt": prompt,
    }
    response = make_api_request(GPT4_URL, payload)
    return response.get("summary", {})

## 3. Main Script Execution

In [6]:
# name: create
# properties: {"require_arg": {"arg_name": {"type": "string", "description": "The url of the webpage to crawl."}}}

# Run the extraction
api_metadata_dicts = scan_repos_for_api(debug=False)



DEBUG: scheduler_post_sleep


In [13]:
# Correctly load the JSON data from the file
with open("tools/scheduler_post_sleep.json", "r") as f:
    tool_node = json.load(f)

connection = neo4j.GraphDatabase.driver(
    "neo4j+s://neo4j.prod.dart.prv.vai-compute.com:7687",
    auth=(os.environ['VAI_NEO4J_USER'], os.environ['VAI_NEO4J_PASSWORD']), database="vaiflowrory"
)
# connection = neo4j.GraphDatabase.driver(
#     "neo4j+s://neo4j.prod.dart.prv.vai-compute.com:7687",
#     auth=(os.environ['VAI_NEO4J_USER'], os.environ['VAI_NEO4J_PASSWORD']), database="alwaysbeclosing"
#     auth=(os.environ['VAI_NEO4J_USER'], os.environ['VAI_NEO4J_PASSWORD']), database="vaidemo"
# )

# Remove uuid from the update props to the UUID on the node doesn't change
create_props = tool_node.copy()
# tool_node.pop("uuid", None)
update_props =  tool_node

# Serialize nested properties into JSON strings
create_props_serialized = {
    key: (json.dumps(value) if isinstance(value, dict) else value)
    for key, value in create_props.items()
}

update_props_serialized = {
    key: (json.dumps(value) if isinstance(value, dict) else value)
    for key, value in update_props.items()
}

# Run the Neo4j query with serialized properties
with connection.session() as session:
    session.run(
        """
        MERGE (t:Tool {tool_name: $tool_name})
        ON CREATE SET t += $create_props
        ON MATCH SET t += $update_props
        """,
        {
            "tool_name": tool_node["tool_name"],
            "create_props": create_props_serialized,
            "update_props": update_props_serialized,
        },
    )